# Smart Traffic v2.0 — PC4
## Sistema multiagente para control de semáforos en malla 2×2
#
Este notebook implementa una simulación simplificada de tráfico sobre una red de 4 intersecciones dispuestas en cuadrícula. Cada intersección es controlada por un **sub-agente Q-Learning** que decide cuándo mantener o cambiar la fase del semáforo. Un **agente central** coordina a los cuatro mediante una preferencia de fase global (ola verde).
#
Características:
- Modelo celular tipo Nagel–Schreckenberg (sin OBB/SAT/IDM/bézier).
- Sin accidentes: los autos no pueden ocupar la misma celda.
- 4 intersecciones en malla 2×2 con tráfico entre ellas.
- Visualización en matplotlib: estática y animación.


In [ ]:
!pip install -q numpy matplotlib pillow


## Arquitectura multiagente
#
- **Sub-agente** (uno por intersección `Iij`): observa colas horizontales/verticales, fase actual y preferencia del agente central; acciona `{mantener, cambiar}`.
- **Agente central**: observa colas globales y la fase mayoritaria; elige una preferencia de fase `{H, V}` que modifica la recompensa de los sub-agentes, premiando la sincronización en una arteria.


## 1. Imports


In [ ]:
import random, math, json, os
from collections import defaultdict, deque
from typing import List, Tuple, Dict, Optional

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation, PillowWriter
from IPython.display import HTML, display, clear_output

# Reproducibilidad
SEED = 42
random.seed(SEED)
np.random.seed(SEED)


## 2. Configuración e hiperparámetros


In [ ]:
# ---------- Topología ----------
N_INTER_X = 2
N_INTER_Y = 2
CELLS_PER_ARTERY = 35   # celdas entre bordes/intersecciones
VMAX = 5
P_SLOW = 0.15
SPAWN_RATE = 0.25       # probabilidad de spawn por borde por step

# ---------- Entrenamiento ----------
MAX_STEPS = 600
N_EPISODES = 40
T_DECISION = 8          # cada cuántos pasos decide cada sub-agente
T_COORD = 16            # cada cuántos pasos decide el agente central
YELLOW_STEPS = 3        # pasos de transición al cambiar (amarillo)

# ---------- Q-Learning ----------
ALPHA = 0.10
GAMMA = 0.90
EPSILON_START = 1.0
EPSILON_END = 0.05
EPSILON_DECAY = 0.95
ALIGNMENT_REWARD = 5.0

# ---------- Buckets de cola ----------
BUCKETS = [0, 2, 5, 999]   # 4 buckets: 0 / 1-2 / 3-5 / 6+


## 3. Funciones auxiliares


In [ ]:
def bucket(q: int) -> int:
    for i, b in enumerate(BUCKETS):
        if q <= b:
            return i
    return len(BUCKETS) - 1


def is_intersection(n: str) -> bool:
    return n.startswith("I")


def ij(node: str):
    return int(node[1]), int(node[2])


def dir_between(frm: str, to: str) -> str:
    """Dirección cardinal del segmento frm -> to."""
    if is_intersection(frm) and is_intersection(to):
        x1, y1 = ij(frm)
        x2, y2 = ij(to)
        if x2 == x1 + 1:
            return "E"
        if x2 == x1 - 1:
            return "W"
        if y2 == y1 + 1:
            return "S"
        if y2 == y1 - 1:
            return "N"
    # De intersección a borde
    if is_intersection(frm):
        return to[-1]
    # De borde a intersección: dirección opuesta a la etiqueta del borde
    return {"N": "S", "S": "N", "E": "W", "W": "E"}[frm[-1]]


## 4. Clases del modelo (Road, Car, Intersection)


In [ ]:
class Road:
    def __init__(self, frm: str, to: str, direction: str, length: int):
        self.frm = frm
        self.to = to
        self.direction = direction
        self.length = length
        self.cells: List[Optional[Car]] = [None] * length

    def gap_ahead(self, pos: int) -> int:
        g = 0
        for k in range(pos + 1, self.length):
            if self.cells[k] is not None:
                break
            g += 1
        return g

    def __repr__(self):
        return f"Road({self.frm}->{self.to})"


class Car:
    _counter = 0

    def __init__(self, route: List[str]):
        Car._counter += 1
        self.id = Car._counter
        self.route = route
        self.leg = 0
        self.pos = 0
        self.v = 0
        self.wait = 0
        self.moved = False
        self.done = False

    def current_dir(self, roads: Dict[Tuple[str, str], Road]) -> str:
        return roads[(self.route[self.leg], self.route[self.leg + 1])].direction


class Intersection:
    def __init__(self, name: str):
        self.name = name
        self.phase = "H"
        self.yellow = 0

    def is_green(self, direction: str) -> bool:
        if self.yellow > 0:
            return False
        return direction in ("E", "W") if self.phase == "H" else direction in ("N", "S")

    def apply_action(self, action: int):
        """action: 0=mantener, 1=cambiar."""
        if self.yellow > 0:
            self.yellow -= 1
            if self.yellow == 0:
                self.phase = "V" if self.phase == "H" else "H"
            return
        if action == 1:
            self.yellow = YELLOW_STEPS


## 5. Agentes Q-Learning


In [ ]:
class QAgent:
    def __init__(self, actions: List[int], alpha: float = ALPHA,
                 gamma: float = GAMMA, eps: float = EPSILON_START):
        self.Q = defaultdict(float)
        self.actions = actions
        self.alpha = alpha
        self.gamma = gamma
        self.epsilon = eps

    def act(self, state: Tuple) -> int:
        if random.random() < self.epsilon:
            return random.choice(self.actions)
        qs = [self.Q[(state, a)] for a in self.actions]
        m = max(qs)
        best = [a for a, q in zip(self.actions, qs) if q == m]
        return random.choice(best)

    def update(self, state: Tuple, action: int, reward: float,
               next_state: Tuple, done: bool):
        key = (state, action)
        if done:
            target = reward
        else:
            target = reward + self.gamma * max(self.Q[(next_state, a)] for a in self.actions)
        self.Q[key] += self.alpha * (target - self.Q[key])

    def decay(self, factor: float = EPSILON_DECAY):
        self.epsilon = max(EPSILON_END, self.epsilon * factor)


class SpacedQAgent(QAgent):
    def __init__(self, actions: List[int], period: int):
        super().__init__(actions)
        self.period = period
        self.pending: Optional[Tuple[Tuple, int]] = None
        self.acc_reward = 0.0

    def should_decide(self, step: int) -> bool:
        return step % self.period == 0

    def decide(self, state: Tuple) -> int:
        action = self.act(state)
        self.pending = (state, action)
        self.acc_reward = 0.0
        return action

    def finish_period(self, next_state: Tuple, done: bool):
        if self.pending is None:
            return
        s, a = self.pending
        self.update(s, a, self.acc_reward, next_state, done)
        self.pending = None

    def add_reward(self, r: float):
        self.acc_reward += r


## 6. Simulador de red 2×2


In [ ]:
class Simulador:
    def __init__(self):
        self.intersections: Dict[str, Intersection] = {}
        self.roads: Dict[Tuple[str, str], Road] = {}
        self.cars: List[Car] = []
        self.sources: List[str] = []
        self.sinks: List[str] = []
        self.step_count = 0
        self.metrics = {
            "throughput": 0,
            "total_wait": 0,
            "queue_global": [],
        }
        self._build_network()

    def _build_network(self):
        # Intersecciones
        for i in range(N_INTER_X):
            for j in range(N_INTER_Y):
                name = f"I{i}{j}"
                self.intersections[name] = Intersection(name)
        # Aristas internas horizontales
        for j in range(N_INTER_Y):
            for i in range(N_INTER_X - 1):
                a, b = f"I{i}{j}", f"I{i+1}{j}"
                self.roads[(a, b)] = Road(a, b, "E", CELLS_PER_ARTERY)
                self.roads[(b, a)] = Road(b, a, "W", CELLS_PER_ARTERY)
        # Aristas internas verticales
        for i in range(N_INTER_X):
            for j in range(N_INTER_Y - 1):
                a, b = f"I{i}{j}", f"I{i}{j+1}"
                self.roads[(a, b)] = Road(a, b, "S", CELLS_PER_ARTERY)
                self.roads[(b, a)] = Road(b, a, "N", CELLS_PER_ARTERY)
        # Bordes externos
        for name in self.intersections:
            i, j = ij(name)
            for d in ["N", "S", "E", "W"]:
                ext = f"E{i}{j}_{d}"
                self.sources.append(ext)
                self.sinks.append(ext)
                inc_dir = {"N": "S", "S": "N", "E": "W", "W": "E"}[d]
                self.roads[(ext, name)] = Road(ext, name, inc_dir, CELLS_PER_ARTERY)
                self.roads[(name, ext)] = Road(name, ext, d, CELLS_PER_ARTERY)

    def reset(self):
        self.cars = []
        self.step_count = 0
        for r in self.roads.values():
            r.cells = [None] * r.length
        for inter in self.intersections.values():
            inter.phase = "H"
            inter.yellow = 0
        self.metrics = {"throughput": 0, "total_wait": 0, "queue_global": []}
        Car._counter = 0

    def shortest_path(self, src: str, dst: str) -> List[str]:
        if src == dst:
            return [src]
        q = deque([(src, [src])])
        visited = {src}
        while q:
            node, path = q.popleft()
            for (a, b) in self.roads:
                if a == node and b not in visited:
                    visited.add(b)
                    npath = path + [b]
                    if b == dst:
                        return npath
                    q.append((b, npath))
        return [src, dst]

    def spawn(self):
        for src in self.sources:
            if random.random() >= SPAWN_RATE:
                continue
            dst = random.choice(self.sinks)
            if dst == src:
                continue
            route = self.shortest_path(src, dst)
            if len(route) < 2:
                continue
            road = self.roads[(route[0], route[1])]
            if road.cells[0] is not None:
                continue
            car = Car(route)
            road.cells[0] = car
            self.cars.append(car)

    def queues(self, inter: Intersection) -> Tuple[int, int]:
        qH = qV = 0
        for (a, b), road in self.roads.items():
            if b != inter.name:
                continue
            for k in range(max(0, road.length - 3), road.length):
                car = road.cells[k]
                if car is not None and car.v == 0:
                    if road.direction in ("E", "W"):
                        qH += 1
                    else:
                        qV += 1
        return qH, qV

    def total_queue(self) -> int:
        return sum(qH + qV for inter in self.intersections.values() for qH, qV in [self.queues(inter)])

    def neighbor_queues(self, inter: Intersection) -> Tuple[int, int]:
        x, y = ij(inter.name)
        nH = nV = 0
        count = 0
        for name in self.intersections:
            if name == inter.name:
                continue
            x2, y2 = ij(name)
            if x2 == x or y2 == y:
                qh, qv = self.queues(self.intersections[name])
                nH += qh
                nV += qv
                count += 1
        if count == 0:
            return 0, 0
        return nH // count, nV // count

    def state_sub(self, inter: Intersection, preferred: str) -> Tuple:
        qH, qV = self.queues(inter)
        nH, nV = self.neighbor_queues(inter)
        return (bucket(qH), bucket(qV),
                0 if inter.phase == "H" else 1,
                bucket(nH), bucket(nV),
                0 if preferred == "H" else 1)

    def state_central(self) -> Tuple:
        totalH = totalV = 0
        h_count = v_count = 0
        for inter in self.intersections.values():
            qh, qv = self.queues(inter)
            totalH += qh
            totalV += qv
            if inter.phase == "H":
                h_count += 1
            else:
                v_count += 1
        majority = 0 if h_count >= v_count else 1
        return (bucket(totalH), bucket(totalV), majority)

    def apply_actions(self, actions: Dict[str, int]):
        for name, inter in self.intersections.items():
            inter.apply_action(actions.get(name, 0))

    def _resolve_crossing(self, car: Car, road: Road):
        inter_name = road.to
        if not is_intersection(inter_name):
            road.cells[car.pos] = None
            car.done = True
            self.metrics["throughput"] += 1
            self.metrics["total_wait"] += car.wait
            return

        inter = self.intersections[inter_name]
        next_node = car.route[car.leg + 2]
        out_dir = dir_between(inter_name, next_node)
        if not inter.is_green(out_dir):
            car.v = 0
            return
        next_road = self.roads[(inter_name, next_node)]
        if next_road.cells[0] is not None:
            car.v = 0
            return
        road.cells[car.pos] = None
        next_road.cells[0] = car
        car.pos = 0
        car.leg += 1
        car.moved = True

    def step_physics(self):
        for car in self.cars:
            car.moved = False

        for road in self.roads.values():
            for pos in range(road.length - 1, -1, -1):
                car = road.cells[pos]
                if car is None or car.moved or car.done:
                    continue

                # Auto detenido en la línea de parada: intenta cruzar
                if car.pos == road.length - 1:
                    self._resolve_crossing(car, road)
                    if not car.moved:
                        car.v = 0
                        car.moved = True
                        car.wait += 1
                    continue

                car.v = min(car.v + 1, VMAX)
                gap = road.gap_ahead(car.pos)
                car.v = min(car.v, gap)
                if random.random() < P_SLOW and car.v > 0:
                    car.v -= 1

                new_pos = car.pos + car.v
                if new_pos >= road.length:
                    self._resolve_crossing(car, road)
                else:
                    road.cells[car.pos] = None
                    road.cells[new_pos] = car
                    car.pos = new_pos
                    car.moved = True

                if car.v == 0:
                    car.wait += 1

        self.cars = [c for c in self.cars if not c.done]
        self.step_count += 1


## 7. Función de entrenamiento


In [ ]:
def train_agents(sim: Simulador, n_episodes: int = N_EPISODES,
                 max_steps: int = MAX_STEPS):
    sub_agents = {name: SpacedQAgent([0, 1], T_DECISION)
                  for name in sim.intersections}
    central = SpacedQAgent([0, 1], T_COORD)

    history = {
        "episode": [],
        "reward_sub": [],
        "reward_central": [],
        "throughput": [],
        "avg_wait": [],
        "epsilon_sub": [],
        "epsilon_central": [],
    }

    preferred_phase = "H"

    for ep in range(n_episodes):
        sim.reset()
        ep_reward_sub = 0.0
        ep_reward_central = 0.0

        for step in range(max_steps):
            sim.spawn()

            # Agente central
            if central.should_decide(step):
                if central.pending is not None:
                    ep_reward_central += central.acc_reward
                    central.finish_period(sim.state_central(), step == max_steps - 1)
                action_c = central.decide(sim.state_central())
                preferred_phase = "H" if action_c == 0 else "V"

            # Sub-agentes
            actions = {}
            for name, inter in sim.intersections.items():
                agent = sub_agents[name]
                if agent.should_decide(step):
                    if agent.pending is not None:
                        ep_reward_sub += agent.acc_reward
                        agent.finish_period(sim.state_sub(inter, preferred_phase),
                                            step == max_steps - 1)
                    action = agent.decide(sim.state_sub(inter, preferred_phase))
                    actions[name] = action
                else:
                    actions[name] = 0

            sim.apply_actions(actions)
            sim.step_physics()

            central.add_reward(-sim.total_queue())
            for name, inter in sim.intersections.items():
                qh, qv = sim.queues(inter)
                r = -(qh + qv)
                if inter.phase == preferred_phase:
                    r += ALIGNMENT_REWARD
                else:
                    r -= ALIGNMENT_REWARD
                sub_agents[name].add_reward(r)

        # Cierre de episodio
        for name, inter in sim.intersections.items():
            agent = sub_agents[name]
            if agent.pending is not None:
                ep_reward_sub += agent.acc_reward
                agent.finish_period(sim.state_sub(inter, preferred_phase), True)
        if central.pending is not None:
            ep_reward_central += central.acc_reward
            central.finish_period(sim.state_central(), True)

        throughput = sim.metrics["throughput"]
        avg_wait = sim.metrics["total_wait"] / max(1, throughput)
        history["episode"].append(ep + 1)
        history["reward_sub"].append(ep_reward_sub / len(sub_agents))
        history["reward_central"].append(ep_reward_central)
        history["throughput"].append(throughput)
        history["avg_wait"].append(avg_wait)
        history["epsilon_sub"].append(sub_agents[list(sub_agents.keys())[0]].epsilon)
        history["epsilon_central"].append(central.epsilon)

        for agent in sub_agents.values():
            agent.decay(EPSILON_DECAY)
        central.decay(EPSILON_DECAY)

    return sub_agents, central, history


## 8. Ejecutar entrenamiento


In [ ]:
sim = Simulador()
sub_agents, central, history = train_agents(sim)
print("Entrenamiento finalizado.")
print(f"Throughput último episodio: {history['throughput'][-1]}")
print(f"Espera promedio último episodio: {history['avg_wait'][-1]:.2f}")


## 9. Gráficos de métricas


In [ ]:
fig, axes = plt.subplots(2, 2, figsize=(12, 9))

ax = axes[0, 0]
ax.plot(history["episode"], history["reward_sub"], label="Sub-agente promedio")
ax.plot(history["episode"], history["reward_central"], label="Agente central")
ax.set_xlabel("Episodio")
ax.set_ylabel("Recompensa acumulada")
ax.set_title("Recompensa por episodio")
ax.legend()
ax.grid(True)

ax = axes[0, 1]
ax.plot(history["episode"], history["throughput"], color="green")
ax.set_xlabel("Episodio")
ax.set_ylabel("Autos completados")
ax.set_title("Throughput")
ax.grid(True)

ax = axes[1, 0]
ax.plot(history["episode"], history["avg_wait"], color="red")
ax.set_xlabel("Episodio")
ax.set_ylabel("Pasos de espera")
ax.set_title("Espera promedio por auto")
ax.grid(True)

ax = axes[1, 1]
ax.plot(history["episode"], history["epsilon_sub"], label="Sub-agente")
ax.plot(history["episode"], history["epsilon_central"], label="Central")
ax.set_xlabel("Episodio")
ax.set_ylabel("Epsilon")
ax.set_title("Decaimiento de exploración")
ax.legend()
ax.grid(True)

plt.tight_layout()
plt.show()


## 10. Visualización de una instantánea


In [ ]:
def coords(node: str):
    if is_intersection(node):
        x, y = ij(node)
        return x * 2, y * 2
    x, y = ij(node[:-2])
    d = node[-1]
    if d == "N":
        return x * 2, y * 2 + 1.5
    if d == "S":
        return x * 2, y * 2 - 1.5
    if d == "E":
        return x * 2 + 1.5, y * 2
    if d == "W":
        return x * 2 - 1.5, y * 2


def draw_network(sim: Simulador, ax=None):
    if ax is None:
        fig, ax = plt.subplots(figsize=(8, 8))
    ax.clear()
    ax.set_xlim(-1, N_INTER_X * 2 + 1)
    ax.set_ylim(-1, N_INTER_Y * 2 + 1)
    ax.set_aspect("equal")
    ax.axis("off")

    for name, inter in sim.intersections.items():
        x, y = ij(name)
        ax.plot(x * 2, y * 2, "o", markersize=24, color="lightgray", zorder=2)
        color_h = "green" if inter.is_green("E") else "red"
        color_v = "green" if inter.is_green("N") else "red"
        ax.plot(x * 2 + 0.35, y * 2, "o", color=color_h, markersize=8, zorder=3)
        ax.plot(x * 2, y * 2 + 0.35, "o", color=color_v, markersize=8, zorder=3)

    for road in sim.roads.values():
        x1, y1 = coords(road.frm)
        x2, y2 = coords(road.to)
        for k, car in enumerate(road.cells):
            if car is None:
                continue
            t = k / max(1, road.length - 1)
            cx = x1 + t * (x2 - x1)
            cy = y1 + t * (y2 - y1)
            color = "blue" if road.direction in ("N", "S") else "red"
            ax.plot(cx, cy, "s", markersize=6, color=color, zorder=4)

    ax.set_title(f"Paso {sim.step_count} | Cola total {sim.total_queue()}")
    return ax


draw_network(sim)
plt.show()


## 11. Animación de la simulación


In [ ]:
from IPython.display import HTML, Image

# Reutilizamos los agentes ya entrenados (sub_agents, central) de la celda de entrenamiento.
sim_anim = Simulador()
sim_anim.reset()


def precompute_frames(sim, sub_agents, central_agent,
                      n_frames=300, sub_steps=3):
    """Precomputa los estados serializables de la simulacion."""
    frames = []
    for _ in range(n_frames):
        for _ in range(sub_steps):
            sim.spawn()
            preferred = "H" if central_agent.act(sim.state_central()) == 0 else "V"
            actions = {name: sub_agents[name].act(sim.state_sub(inter, preferred))
                       for name, inter in sim.intersections.items()}
            sim.apply_actions(actions)
            sim.step_physics()
        intersections = []
        for name, inter in sim.intersections.items():
            x, y = ij(name)
            intersections.append([x, y,
                                  1 if inter.is_green("E") else 0,
                                  1 if inter.is_green("N") else 0])
        cars_state = []
        for road in sim.roads.values():
            x1, y1 = coords(road.frm)
            x2, y2 = coords(road.to)
            is_h = 1 if road.direction in ("E", "W") else 0
            for k, car in enumerate(road.cells):
                if car is None:
                    continue
                t = k / max(1, road.length - 1)
                cx = x1 + t * (x2 - x1)
                cy = y1 + t * (y2 - y1)
                cars_state.append([round(cx, 3), round(cy, 3), is_h])
        frames.append({"inters": intersections, "cars": cars_state,
                       "q": sim.total_queue(), "step": sim.step_count})
    return frames


frames_data = precompute_frames(sim_anim, sub_agents, central, n_frames=300, sub_steps=3)
print(f"Frames precomputados: {len(frames_data)}")


def render_canvas_html(frames, width=560, height=560):
    import json as _json
    data = _json.dumps(frames)
    scale = width / 5.0
    ox = 1.5 * scale
    oy = 1.5 * scale
    nmax = max(0, len(frames) - 1)
    template = '''<div style="display:flex;flex-direction:column;align-items:center;">
  <div>
    <button id="pc4_play" style="font-size:14px;padding:4px 12px;">Play</button>
    <span id="pc4_info" style="margin-left:12px;font-family:monospace;">Paso 0 | Cola 0</span>
  </div>
  <input type="range" id="pc4_scrub" min="0" max="__NMAX__" value="0"
         style="width:__WIDTH__px;margin:8px 0;" />
  <canvas id="pc4_canvas" width="__WIDTH__" height="__HEIGHT__"
          style="border:1px solid #888;background:#222;"></canvas>
</div>
<script id="pc4_data" type="application/json">__DATA__</script>
<script>
(function(){
  const data = JSON.parse(document.getElementById('pc4_data').textContent);
  const c = document.getElementById('pc4_canvas');
  const ctx = c.getContext('2d');
  const W = c.width, H = c.height;
  const SCALE = __SCALE__;
  const OX = __OX__, OY = __OY__;
  function tx(v){ return OX + v * SCALE; }
  function ty(v){ return H - (OY + v * SCALE); }
  const playBtn = document.getElementById('pc4_play');
  const scrub = document.getElementById('pc4_scrub');
  const info = document.getElementById('pc4_info');
  let frameIdx = 0;
  let playing = false;
  let rafId = null;
  let lastTime = 0;
  const FPS = 30;
  const INTERVAL = 1000 / FPS;

  function drawArteries(){
    ctx.strokeStyle = '#444';
    ctx.lineWidth = 6;
    for (let j=0;j<2;j++){
      ctx.beginPath();
      ctx.moveTo(tx(-1.5), ty(j*2)); ctx.lineTo(tx(3.5), ty(j*2));
      ctx.stroke();
    }
    for (let i=0;i<2;i++){
      ctx.beginPath();
      ctx.moveTo(tx(i*2), ty(-1.5)); ctx.lineTo(tx(i*2), ty(3.5));
      ctx.stroke();
    }
    ctx.lineWidth = 2;
    ctx.strokeStyle = '#555';
    for (let i=0;i<2;i++){
      for (let j=0;j<2;j++){
        ctx.beginPath(); ctx.moveTo(tx(i*2), ty(j*2)); ctx.lineTo(tx(i*2), ty(j*2+1.5)); ctx.stroke();
        ctx.beginPath(); ctx.moveTo(tx(i*2), ty(j*2)); ctx.lineTo(tx(i*2), ty(j*2-1.5)); ctx.stroke();
        ctx.beginPath(); ctx.moveTo(tx(i*2), ty(j*2)); ctx.lineTo(tx(i*2+1.5), ty(j*2)); ctx.stroke();
        ctx.beginPath(); ctx.moveTo(tx(i*2), ty(j*2)); ctx.lineTo(tx(i*2-1.5), ty(j*2)); ctx.stroke();
      }
    }
  }

  function drawFrame(fr){
    ctx.fillStyle = '#222';
    ctx.fillRect(0,0,W,H);
    drawArteries();
    for (const it of fr.inters){
      const ix = it[0], iy = it[1], gh = it[2], gv = it[3];
      const cx = tx(ix*2), cy = ty(iy*2);
      ctx.fillStyle = '#bbbbbb';
      ctx.beginPath(); ctx.arc(cx, cy, 13, 0, 2*Math.PI); ctx.fill();
      ctx.fillStyle = gh===1 ? '#22ff22' : '#ff2222';
      ctx.beginPath(); ctx.arc(cx+6, cy, 4, 0, 2*Math.PI); ctx.fill();
      ctx.fillStyle = gv===1 ? '#22ff22' : '#ff2222';
      ctx.beginPath(); ctx.arc(cx, cy-6, 4, 0, 2*Math.PI); ctx.fill();
    }
    for (const cr of fr.cars){
      const px = cr[0], py = cr[1], isH = cr[2];
      ctx.fillStyle = isH===1 ? '#ff5555' : '#5599ff';
      ctx.beginPath(); ctx.arc(tx(px), ty(py), 4, 0, 2*Math.PI); ctx.fill();
    }
    info.textContent = 'Paso ' + fr.step + ' | Cola ' + fr.q;
  }

  function step(now){
    if (!playing) return;
    if (now - lastTime >= INTERVAL){
      lastTime = now;
      frameIdx = (frameIdx + 1) % data.length;
      scrub.value = frameIdx;
      drawFrame(data[frameIdx]);
    }
    rafId = requestAnimationFrame(step);
  }

  playBtn.onclick = function(){
    playing = !playing;
    playBtn.textContent = playing ? 'Pausa' : 'Play';
    if (playing){
      lastTime = performance.now();
      rafId = requestAnimationFrame(step);
    } else {
      cancelAnimationFrame(rafId);
    }
  };

  scrub.oninput = function(){
    frameIdx = parseInt(scrub.value);
    drawFrame(data[frameIdx]);
  };

  drawFrame(data[0]);
})();
</script>'''
    html = (template
            .replace("__WIDTH__", str(width))
            .replace("__HEIGHT__", str(height))
            .replace("__DATA__", data)
            .replace("__SCALE__", str(scale))
            .replace("__OX__", str(ox))
            .replace("__OY__", str(oy))
            .replace("__NMAX__", str(nmax)))
    return html


html_str = render_canvas_html(frames_data)
display(HTML(html_str))


## 12. Informe resumido
#
El informe completo se encuentra en `informe.md`. Allí se detalla la arquitectura MAPE, las ecuaciones del Q-Learning, la definición de estados/acciones/recompensas y los resultados esperados.
